In [8]:
#remove when converting to .py file
from pathlib import Path
import importlib.util

PROJECT_ROOT = Path.cwd()
helper_path = PROJECT_ROOT / ".." /"src" / "utils.py"
spec = importlib.util.spec_from_file_location("utils", helper_path)
utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(utils)

In [20]:
import numpy as np
import librosa
import torch
from transformers import pipeline
import os
import pprint
import json
print(np.isnan(0.0))


False


In [10]:
torch.backends.cudnn.enabled = True
if torch.cuda.is_available():
    print("GPU(s):", torch.cuda.device_count(), torch.cuda.get_device_name(0))  #you need cuda otherwise set device to cpu

GPU(s): 1 NVIDIA GeForce RTX 4060 Laptop GPU


In [11]:
audio_classifier = pipeline(task="zero-shot-audio-classification", model="laion/larger_clap_general", batch=8, device='cuda')   #SET IT HERE

Device set to use cuda


In [ ]:
clap_label_json = "../json/clap_labels.json"
with open(clap_label_json, 'r') as f:
    music_labels = json.load(f)

TypeError: load() missing 1 required positional argument: 'fp'

In [14]:
audio_segments_path = '../segments/testSong'

In [15]:
result = []
for segment in os.listdir(audio_segments_path):
    audio = os.path.join(audio_segments_path, segment)
    y, sr = librosa.load(audio, sr=22050)
    features = {}
    for label in music_labels:
        classes = music_labels[label]
        features[label] = audio_classifier(y, candidate_labels = classes)
    result.append(features)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [ ]:
pprint.pprint(result[4]['moods'])

[{'label': 'playful', 'score': 0.2463110089302063},
 {'label': 'tense', 'score': 0.18269051611423492},
 {'label': 'carefree', 'score': 0.11743742972612381},
 {'label': 'exciting', 'score': 0.10803357511758804},
 {'label': 'anxious', 'score': 0.09992749243974686},
 {'label': 'hopeful', 'score': 0.05644002929329872},
 {'label': 'nostalgic', 'score': 0.030870258808135986},
 {'label': 'epic', 'score': 0.026774967089295387},
 {'label': 'scary', 'score': 0.01801704242825508},
 {'label': 'romantic', 'score': 0.015214020386338234},
 {'label': 'love', 'score': 0.01452880259603262},
 {'label': 'happy', 'score': 0.014386937022209167},
 {'label': 'groovy', 'score': 0.013666625134646893},
 {'label': 'uplifting', 'score': 0.012542719021439552},
 {'label': 'sexy', 'score': 0.008897041901946068},
 {'label': 'empowering', 'score': 0.008428600616753101},
 {'label': 'funny', 'score': 0.007909861393272877},
 {'label': 'dramatic', 'score': 0.006011706776916981},
 {'label': 'serious', 'score': 0.00440619513

In [ ]:
utils.save_as_json("clap_results", result)
#Each segment has all the weighted mood and genres. 